<p align="left">
  <img src="https://raw.githubusercontent.com/The-DigitalAcademy/shaper-elearning-images/refs/heads/main/CI/Shaper-Full-Logo-Horizontal.png" alt="Shaper logo" width="250">
</p>

© 2026 Shaper. All rights reserved. This material is confidential and may not be reproduced, distributed, or used without prior written permission.

## Phase 1: Understand the Dataset

In [ ]:
# Import required libraries
import pandas as pd

In [ ]:
# Load the dataset
url = "https://drive.google.com/uc?id=1J9qyOW_GF6qxNLaO6SCzT5QfZQ6XfwF7"
df = pd.read_csv(url)

In [ ]:
# Preview first 5 rows
df.head()

In [ ]:
# Inspect structure, data types, and missing values
df.info()

In [ ]:
# Summary statistics for numerical variables
df.describe()

In [ ]:
# Print the dataset shape (rows, columns)
print(df.shape)

In [ ]:
# Identify numerical variables
numerical_vars = df.select_dtypes(include=["number"]).columns

print("Number of numerical variables:", len(numerical_vars))

In [ ]:
# Identify categorical variables
categorical_vars = df.select_dtypes(include=["object"]).columns

print("Number of categorical variables:", len(categorical_vars))

## Phase 2: Data Quality and Integrity

In [ ]:
# Count missing values per column
missing_counts = df.isna().sum()
missing_counts_sorted = missing_counts.sort_values(ascending=False)
missing_counts_sorted

In [ ]:
# Calculate percentage missing per column
missing_percentages = (df.isna().mean() * 100).sort_values(ascending=False)

missing_percentages

In [ ]:
# Columns with no missing data
no_missing_columns = missing_counts[missing_counts == 0].index.tolist()

print("Columns with no missing data:")
print(no_missing_columns)

In [ ]:
# Column iwth highest missing data
highest_missing_column = missing_counts_sorted.index[0]
highest_missing_count = missing_counts_sorted.iloc[0]
highest_missing_percentage = missing_percentages.loc[highest_missing_column]

print("Column with highest missing data:", highest_missing_column)
print("Missing Count:", highest_missing_count)
print("Missing Percentage:", round(missing_percentages.iloc[0], 2), "%")

## Phase 3: Distribution and Spread

In [ ]:
# Compute summary statistics
summary_stats = df[numerical_vars].describe()
summary_stats

In [ ]:
# Identify variables with highest and lowest spread
std_values = df[numerical_vars].std().sort_values(ascending=False)

high_spread_variable = std_values.index[0]
low_spread_variable = std_values.index[-1]

print("Variable with highest standard deviation:", high_spread_variable)
print("Highest standard deviation value:", std_values.iloc[0])

print("\nVariable with lowest standard deviation:", low_spread_variable)
print("Lowest standard deviation value:", std_values.iloc[-1])

In [ ]:
# Compute skewness for numerical columns
skew_values = df[numerical_vars].skew()

print("=== SKEWNESS VALUES ===")
print(skew_values.sort_values(ascending=False))

# Identify most skewed variable (absolute skew)
abs_skew = skew_values.abs().sort_values(ascending=False)

most_skewed_variable = abs_skew.index[0]
most_skewed_value = skew_values[most_skewed_variable]

print("\nMost skewed variable (by absolute skewness):", most_skewed_variable)
print("Skewness value:", most_skewed_value)

In [ ]:
# Apply IQR method to trip_total

# Select the variable
variable = "trip_total"

# Calculate Q1 and Q3
Q1 = df[variable].quantile(0.25)
Q3 = df[variable].quantile(0.75)

# Calculate IQR
IQR = Q3 - Q1

# Calculate bounds
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

print(f"Q1: {Q1}")
print(f"Q3: {Q3}")
print(f"IQR: {IQR}")
print(f"Lower Bound: {lower_bound}")
print(f"Upper Bound: {upper_bound}")

In [ ]:
# Identify outliers
outliers = df[(df[variable] < lower_bound) | (df[variable] > upper_bound)]

# Count outliers
print("Number of potential outliers:", len(outliers))

In [ ]:
# Percentage of rows flagged as outliers for this variable
outlier_percentage = len(outliers) / len(df) * 100

print(f"For variable '{variable}', {outlier_percentage:.2f}% "
      f"of rows are flagged as potential outliers.")

## Phase 4: Distribution Visualisation

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")

In [ ]:
# Define 99th percentile limits for cleaner visualisation
miles_upper = df["trip_miles"].quantile(0.99)
total_upper = df["trip_total"].quantile(0.99)
duration_upper = df["trip_seconds"].quantile(0.99)

df_vis = df[
    (df["trip_miles"] <= miles_upper) &
    (df["trip_total"] <= total_upper) &
    (df["trip_seconds"] <= duration_upper)
]

In [ ]:
# Histogram: Trip Miles
plt.figure()
plt.hist(df_vis["trip_miles"], bins=30)
plt.title("Distribution of Trip Miles (Below 99th Percentile)")
plt.xlabel("Trip Miles")
plt.ylabel("Frequency")
plt.show()

In [ ]:
# Histogram: Trip Total
plt.figure()
plt.hist(df_vis["trip_total"], bins=30)
plt.title("Distribution of Trip Total (Below 99th Percentile)")
plt.xlabel("Trip Total ($)")
plt.ylabel("Frequency")
plt.show()

In [ ]:
plt.figure()
plt.hist(df_vis["trip_seconds"], bins=30)
plt.title("Distribution of Trip Duration (Below 99th Percentile)")
plt.xlabel("Trip Duration (Seconds)")
plt.ylabel("Frequency")
plt.show()

In [ ]:
# Boxplot: Trip Total
plt.figure()
plt.boxplot(df_vis["trip_total"], vert=False)
plt.title("Boxplot of Trip Total")
plt.xlabel("Trip Total ($)")
plt.show()

## Phase 5: Transformations and Segmentation

In [ ]:
# Remove zero-mile trips to avoid division errors
df = df[df["trip_miles"] > 0]

# Derived variable: Fare per mile
df["fare_per_mile"] = df["fare"] / df["trip_miles"]

print("Derived variable created: fare_per_mile")
df[["fare", "trip_miles", "fare_per_mile"]].head()

In [ ]:
# Create distance bands
distance_bins = [0, 1, 5, 10, 20, df["trip_miles"].max()]
distance_labels = ["Very Short", "Short", "Medium", "Long", "Very Long"]

df["distance_band"] = pd.cut(
    df["trip_miles"],
    bins=distance_bins,
    labels=distance_labels
)

print("Binned variable created: distance_band")
df[["trip_miles", "distance_band"]].head()

In [ ]:
# Payment type summary
payment_summary = (
    df.groupby("payment_type")["trip_total"]
    .agg(["count", "mean", "median", "std"])
    .round(2)
)

payment_summary

In [ ]:
# Average trip total by distance band
band_summary = df.groupby("distance_band")["trip_total"].mean().round(2)
band_summary

In [ ]:
# Stacked revenue composition
band_revenue_components = (
    df.groupby("distance_band")[["fare", "tolls", "extras", "tips"]]
    .mean()
    .round(2)
)

band_revenue_components.plot(kind="bar", stacked=True)
plt.title("Average Revenue Composition by Distance Band")
plt.show()

## Phase 6: Relationships and Interaction

In [ ]:
# Create correlation matrix for numerical variables
numeric_df = df.select_dtypes(include=["int64", "float64"])
correlation_matrix = numeric_df.corr()

correlation_matrix.round(3)

In [ ]:
# Create heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(correlation_matrix, cmap="coolwarm", center=0)
plt.title("Correlation Heatmap")
plt.show()

In [ ]:
# Filtered dataset for visual clarity
miles_upper = df["trip_miles"].quantile(0.99)
total_upper = df["trip_total"].quantile(0.99)

df_vis = df[
    (df["trip_miles"] <= miles_upper) &
    (df["trip_total"] <= total_upper)
]

In [ ]:
# Scatter plot for trip total versus trip miles
sns.scatterplot(
    data=df_vis,
    x="trip_miles",
    y="trip_total",
    alpha=0.4
)
plt.title("Trip Total vs Trip Miles")
plt.show()

In [ ]:
# Scatter with trend line
sns.regplot(
    data=df_vis,
    x="trip_miles",
    y="trip_total",
    scatter_kws={"alpha": 0.4},
    line_kws={"color": "red"}
)

plt.title("Trip Total vs Trip Miles")
plt.show()

In [ ]:
# Faceted by payment type
g = sns.FacetGrid(df_vis, col="payment_type", col_wrap=3)
g.map_dataframe(sns.scatterplot, x="trip_miles", y="trip_total", alpha=0.5)